In [50]:
# Trabajo Final - Prompt Engineering en IA

# Este notebook desarrolla los dos ejercicios prácticos del módulo de Prompt Engineering.

## Ejercicio 1: Extracción de información de alojamientos de Airbnb

#  Se utilizan descripciones y comentarios de alojamientos para extraer información estructurada mediante LLMs.

## Ejercicio 2: Análisis de reseñas de videojuegos con LLMs

# Se diseña un pipeline en dos pasos:

# 1. Filtrado de reseñas relevantes.
# 2. Extracción de información estructurada a partir de las reseñas seleccionadas.

# El objetivo principal es aplicar técnicas de Prompt Engineering para transformar texto no estructurado en información útil y organizada.

In [51]:
## 1. Configuración inicial
# En esta sección se importan las librerías necesarias y se definen las rutas principales del proyecto.
# Se utiliza una estructura de carpetas separada para mantener organizados los datos originales, los datos procesados y los resultados finales.

In [52]:
# Importamos las librerías principales del proyecto.
# pathlib se usa para trabajar con rutas de forma segura.
# pandas se utiliza para leer, limpiar y transformar los datasets.

from pathlib import Path
import pandas as pd
import numpy as np
import json
import time
import os
from dotenv import load_dotenv

# Configuración para visualizar mejor los DataFrames en el notebook.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 300)

# Definimos las rutas principales del proyecto.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

# Cargamos variables de entorno desde el archivo .env.
load_dotenv(PROJECT_ROOT / ".env")

print("Ruta del proyecto:", PROJECT_ROOT)
print("Carpeta de datos raw:", DATA_RAW)
print("Carpeta de resultados:", RESULTS_DIR)

Ruta del proyecto: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering
Carpeta de datos raw: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\data\raw
Carpeta de resultados: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\results


In [53]:
# ## 2. Carga de datasets del Ejercicio 1
#
# En esta sección se cargan los datasets proporcionados para el Ejercicio 1.
# El primer archivo contiene información de alojamientos de Airbnb en Oporto.
# El segundo archivo contiene comentarios de usuarios sobre alojamientos.
#
# Antes de diseñar prompts o llamar a un LLM, es necesario revisar las columnas reales
# de cada dataset para saber exactamente qué información tenemos disponible.

listings_path = DATA_RAW / "listings1_cleaned.csv"
reviews_path = DATA_RAW / "Airbnb_reviews_5000.csv"

df_listings = pd.read_csv(listings_path)
df_reviews = pd.read_csv(reviews_path)

print("Dataset listings1_cleaned.csv")
print("Filas:", df_listings.shape[0])
print("Columnas:", df_listings.shape[1])
print()

print("Dataset Airbnb_reviews_5000.csv")
print("Filas:", df_reviews.shape[0])
print("Columnas:", df_reviews.shape[1])

Dataset listings1_cleaned.csv
Filas: 8140
Columnas: 37

Dataset Airbnb_reviews_5000.csv
Filas: 5000
Columnas: 8


In [54]:
# ## Nota sobre los datasets utilizados
#
# El enunciado menciona que el archivo listings1_cleaned.csv contiene alojamientos de Airbnb en Oporto.
# Sin embargo, al revisar el contenido real del archivo disponible, aparecen ubicaciones de Montreal.
#
# En este trabajo se utilizan los archivos CSV disponibles con los nombres indicados en el enunciado.
# Esta diferencia entre la descripción del enunciado y el contenido real del dataset se documenta aquí
# para dejar constancia de la decisión tomada.
#
# Esta aclaración no afecta al objetivo principal del ejercicio, ya que la tarea consiste en aplicar
# técnicas de Prompt Engineering para extraer información estructurada a partir de texto no estructurado.

In [55]:
# Mostramos las columnas de ambos datasets para identificar los nombres exactos.
# Esto es importante porque el enunciado habla de la columna "descripción",
# pero en el CSV puede aparecer con otro nombre, por ejemplo "description".

print("Columnas de listings1_cleaned.csv:")
print(df_listings.columns.tolist())

print("\nColumnas de Airbnb_reviews_5000.csv:")
print(df_reviews.columns.tolist())

Columnas de listings1_cleaned.csv:
['listing_id', 'listing_url', 'picture_url', 'name', 'description', 'host_id', 'host_name', 'host_since', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_listings_count', 'host_has_profile_pic', 'neighbourhood', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights', 'number_of_reviews', 'review_scores_rating', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'reviews_per_month', 'availability_365', 'has_availability', 'last_review', 'geographical_group']

Columnas de Airbnb_reviews_5000.csv:
['name', 'host_id', 'host_name', 'date', 'reviewer_id', 'reviewer_name', 'comments', 'language']


In [56]:
# ## 3. Ejercicio 1 - Selección de descripciones largas
#
# El enunciado pide seleccionar entre 10 y 20 descripciones con mayor longitud.
# Para este trabajo se seleccionan las 15 descripciones más largas del dataset.
#
# La longitud del texto se calcula contando el número de caracteres de la columna "description".
# También se eliminan valores vacíos para evitar enviar textos sin contenido al modelo.

df_listings_descriptions = df_listings.copy()

df_listings_descriptions["description"] = df_listings_descriptions["description"].fillna("")
df_listings_descriptions["description_length"] = df_listings_descriptions["description"].str.len()

df_top_descriptions = (
    df_listings_descriptions
    .sort_values("description_length", ascending=False)
    .head(15)
    .copy()
)

print("Número de descripciones seleccionadas:", len(df_top_descriptions))
print("Longitud mínima seleccionada:", df_top_descriptions["description_length"].min())
print("Longitud máxima seleccionada:", df_top_descriptions["description_length"].max())

df_top_descriptions[
    [
        "listing_id",
        "name",
        "neighbourhood",
        "property_type",
        "room_type",
        "accommodates",
        "bathrooms",
        "bedrooms",
        "beds",
        "description_length",
        "description",
    ]
].head()

Número de descripciones seleccionadas: 15
Longitud mínima seleccionada: 1000
Longitud máxima seleccionada: 1000


,listing_id,name,neighbourhood,property_type,room_type,accommodates,bathrooms,bedrooms,beds,description_length,description
729,16325175.0,"Discover Little Italy from a Hip, Cozy Tinyhouse",Rosemont-La Petite-Patrie,Tiny home,Entire home/apt,2,1.0,1,1.0,1000,Fix breakfast in a cozy kitchen and dine at a Scandi-style wood table. Recline on a quaint yellow sofa below a skylight window and be steeped in the vibrant charm of this colorful garage-turned-home with its heated concrete floors and eclectic decor. (permit #302976)<br /><br />One of a kind spa...
883,19093836.0,Luxuriate in the Cozy Apartment near Mount Royal,Le Plateau-Mont-Royal,Entire condo,Entire home/apt,5,1.0,2,3.0,1000,"Prep meals in the sleek kitchen with monochrome and marble accents. Crisp, white accents, a luxurious bathroom, and state-of-art appliances enhance the high-end finish of this home.<br /><br />PLEASE NOTE: Parties are strictly prohibited at our property. Guests in violation of our policy will be..."
835,18508404.0,Luxury Historic Montreal 3 Bdrm w/Deck. # 296183,Côte-des-Neiges-Notre-Dame-de-Grâce,Entire rental unit,Entire home/apt,6,1.0,3,3.0,1000,"Indulge in the luxury of classic Montreal architecture in this restored 1920s space featuring original wood details throughout, contemporary furnishings, and a private back patio deck with Weber barbecue. Host friends and family and dine al fresco in the fully stocked kitchen & historic dining r..."
912,19717514.0,"Unwind in Style at a Luxurious, Modern Condo",Le Plateau-Mont-Royal,Entire condo,Entire home/apt,9,2.0,3,3.0,1000,"Kick back on the crisp, white corner sofa at this brand-new, spacious condo with a high-end, Gray marble bathroom. This quiet, cozy space mixes modern and traditional decor that creates a calming, air-conditioned retreat with a luxurious feel. <br /><br />PLEASE NOTE: Parties are strictly prohib..."
1035,22191982.0,Charming Studio - Next to downtown,Rosemont-La Petite-Patrie,Entire rental unit,Entire home/apt,2,1.0,0,1.0,1000,"You will enjoy sleeping in our soft and comfortable Quinn size bed, equipped with clean and nice smelling sheets and soft pillows. Fully equipped bathroom with extra towels. Fully operational kitchen where you will find everything you need to cook nice meals. You will find closely, Maxi ,a groce..."


In [57]:
# Guardamos el subconjunto de descripciones seleccionadas.
# Esto permite reutilizarlo después sin tener que repetir el filtrado.

top_descriptions_path = DATA_PROCESSED / "ejercicio1_top_15_descripciones_airbnb.csv"

df_top_descriptions.to_csv(top_descriptions_path, index=False, encoding="utf-8-sig")

print("Archivo guardado en:", top_descriptions_path)

Archivo guardado en: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\data\processed\ejercicio1_top_15_descripciones_airbnb.csv


In [58]:
# ## 4. Comprobación de la API key de OpenAI
#
# En esta celda cargamos de nuevo el archivo .env desde la ruta del proyecto.
# Después comprobamos si la API key está disponible en el notebook.
# Por seguridad, no mostramos la clave completa.

env_path = PROJECT_ROOT / ".env"

load_dotenv(env_path, override=True)

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    print("API key cargada correctamente.")
    print("Empieza por:", api_key[:7])
else:
    print("El archivo .env existe, pero no contiene OPENAI_API_KEY o está mal escrito.")

API key cargada correctamente.
Empieza por: sk-proj


In [59]:
# Comprobamos dónde está buscando el notebook el archivo .env.
# También comprobamos si el archivo existe realmente en esa ruta.

env_path = PROJECT_ROOT / ".env"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Ruta esperada del .env:", env_path)
print("¿Existe .env?:", env_path.exists())

PROJECT_ROOT: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering
Ruta esperada del .env: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\.env
¿Existe .env?: True


In [60]:
# ## 5. Prueba de conexión con OpenAI
#
# En esta celda configuramos el cliente de OpenAI y hacemos una llamada mínima
# para comprobar que la API responde correctamente.
#
# Se usa una temperatura baja para obtener una respuesta breve y estable.

from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_completion(prompt, model="gpt-4o-mini", temperature=0.2, max_tokens=200):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

respuesta_prueba = get_completion("Responde solo con la palabra OK.")

print(respuesta_prueba)

OK


In [61]:
# ## 6. Ejercicio 1 - Prompt para extracción de entidades de alojamientos
#
# En esta parte se define el prompt que se enviará al LLM.
# El objetivo es extraer información estructurada desde la descripción de cada alojamiento.
#
# Se solicita una salida en JSON para facilitar su posterior conversión a DataFrame.
# También se indica que, si un dato no aparece en el texto, el modelo debe usar "No mencionado".
# Esto reduce el riesgo de que el modelo invente información.

def build_airbnb_entity_extraction_prompt(row):
    prompt = f"""
Eres un sistema experto en extracción de información desde descripciones de alojamientos turísticos.

Tu tarea es analizar la información de un alojamiento y devolver únicamente un objeto JSON válido.

Extrae las siguientes entidades:

- name: nombre del alojamiento
- location: ubicación o barrio
- main_characteristics: lista de características principales
- type: tipo de alojamiento
- size: tamaño del alojamiento
- capacity: capacidad de huéspedes
- key_amenities: lista de comodidades clave
- proximity_highlights: lista de puntos de interés cercanos

Reglas:
1. Devuelve solo JSON válido, sin explicaciones antes ni después.
2. No inventes información.
3. Si un dato no aparece, usa "No mencionado".
4. Las listas deben ser listas JSON.
5. Usa la información estructurada disponible y la descripción textual.

Información estructurada del alojamiento:
- name: {row.get("name", "No mencionado")}
- neighbourhood: {row.get("neighbourhood", "No mencionado")}
- property_type: {row.get("property_type", "No mencionado")}
- room_type: {row.get("room_type", "No mencionado")}
- accommodates: {row.get("accommodates", "No mencionado")}
- bathrooms: {row.get("bathrooms", "No mencionado")}
- bedrooms: {row.get("bedrooms", "No mencionado")}
- beds: {row.get("beds", "No mencionado")}

Descripción:
\"\"\"{row.get("description", "")}\"\"\"

Formato exacto de salida:
{{
  "name": "...",
  "location": "...",
  "main_characteristics": ["...", "..."],
  "type": "...",
  "size": "...",
  "capacity": "...",
  "key_amenities": ["...", "..."],
  "proximity_highlights": ["...", "..."]
}}
"""
    return prompt

In [62]:
# ## 7. Ejercicio 1 - Prueba del prompt con una descripción
#
# Antes de procesar todas las descripciones seleccionadas, probamos el prompt con una sola fila.
# Esto permite verificar que el LLM devuelve un JSON válido y con la estructura esperada.
#
# Esta prueba reduce errores antes de lanzar varias llamadas a la API.

test_row = df_top_descriptions.iloc[0]

test_prompt = build_airbnb_entity_extraction_prompt(test_row)

test_response = get_completion(
    prompt=test_prompt,
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=800
)

print(test_response)

{
  "name": "Discover Little Italy from a Hip, Cozy Tinyhouse",
  "location": "Rosemont-La Petite-Patrie",
  "main_characteristics": ["cozy kitchen", "heated concrete floors", "eclectic decor", "2 floors", "20' ceiling in the loft"],
  "type": "Tiny home",
  "size": "No mencionado",
  "capacity": "2",
  "key_amenities": ["skylight window", "outdoor patio"],
  "proximity_highlights": ["Jean Talon market", "coffee shops", "cafes", "restaurants", "churches"]
}


In [63]:
# ## 8. Ejercicio 1 - Extracción de entidades para las 15 descripciones
#
# En esta celda se aplica el prompt de extracción a las 15 descripciones seleccionadas.
# Se guarda también el listing_id original para poder relacionar cada resultado con su alojamiento.
#
# Se usa time.sleep() entre llamadas para evitar enviar peticiones demasiado seguidas a la API.
# Aunque el volumen es pequeño, esta práctica ayuda a gestionar posibles límites de uso o rate limits.

airbnb_entities_results = []

for index, row in df_top_descriptions.iterrows():
    print(f"Procesando alojamiento {len(airbnb_entities_results) + 1} de {len(df_top_descriptions)}...")
    
    prompt = build_airbnb_entity_extraction_prompt(row)
    
    response = get_completion(
        prompt=prompt,
        model="gpt-4o-mini",
        temperature=0.2,
        max_tokens=800
    )
    
    try:
        parsed_response = json.loads(response)
        parsed_response["listing_id"] = row["listing_id"]
        airbnb_entities_results.append(parsed_response)
    except json.JSONDecodeError:
        print("Error al convertir la respuesta a JSON en listing_id:", row["listing_id"])
        print(response)
    
    time.sleep(1)

print("Extracciones completadas:", len(airbnb_entities_results))

Procesando alojamiento 1 de 15...
Procesando alojamiento 2 de 15...
Procesando alojamiento 3 de 15...
Procesando alojamiento 4 de 15...
Procesando alojamiento 5 de 15...
Procesando alojamiento 6 de 15...
Procesando alojamiento 7 de 15...
Procesando alojamiento 8 de 15...
Procesando alojamiento 9 de 15...
Procesando alojamiento 10 de 15...
Procesando alojamiento 11 de 15...
Procesando alojamiento 12 de 15...
Procesando alojamiento 13 de 15...
Procesando alojamiento 14 de 15...
Procesando alojamiento 15 de 15...
Extracciones completadas: 15


In [64]:
# ## 9. Ejercicio 1 - Conversión de resultados a DataFrame
#
# Las respuestas del LLM se han guardado como una lista de diccionarios.
# En esta celda se convierten en un DataFrame para poder visualizar mejor el resultado.
#
# Este DataFrame representa la salida estructurada de la primera parte del Ejercicio 1.

df_airbnb_entities = pd.DataFrame(airbnb_entities_results)

# Reordenamos columnas para dejar listing_id al principio.
columns_order = [
    "listing_id",
    "name",
    "location",
    "main_characteristics",
    "type",
    "size",
    "capacity",
    "key_amenities",
    "proximity_highlights"
]

df_airbnb_entities = df_airbnb_entities[columns_order]

df_airbnb_entities.head()

,listing_id,name,location,main_characteristics,type,size,capacity,key_amenities,proximity_highlights
0,16325175.0,"Discover Little Italy from a Hip, Cozy Tinyhouse",Rosemont-La Petite-Patrie,"[cozy kitchen, heated concrete floors, eclectic decor, 2 floors, 20' ceiling in the loft]",Tiny home,No mencionado,2,"[skylight window, outdoor patio]","[Jean Talon market, coffee shops, cafes, restaurants, churches]"
1,19093836.0,Luxuriate in the Cozy Apartment near Mount Royal,Le Plateau-Mont-Royal,"[sleek kitchen with monochrome and marble accents, luxurious bathroom, state-of-art appliances, private balcony, Jacuzzi tub]",Entire condo,No mencionado,5,"[stainless steel appliances, granite countertops, TV with Netflix, couch]","[bakery, supermarket]"
2,18508404.0,Luxury Historic Montreal 3 Bdrm w/Deck. # 296183,Côte-des-Neiges-Notre-Dame-de-Grâce,"[restored 1920s space, original wood details, private back patio deck]",Entire rental unit,No mencionado,6,"[Weber barbecue, high speed Fiber optic internet, SmartTV, HEPA Air purifier, EV charging, parking]","[Netflix, Disney + channel, board games, books]"
3,19717514.0,"Unwind in Style at a Luxurious, Modern Condo",Le Plateau-Mont-Royal,"[brand-new, spacious, high-end, luxurious feel, mixes modern and traditional decor]",Entire condo,No mencionado,9,"[air conditioning, fully equipped modern kitchen, stainless steel appliances, granite countertops]",[No mencionado]
4,22191982.0,Charming Studio - Next to downtown,Rosemont-La Petite-Patrie,"[Quinn size bed, Fully equipped bathroom, Fully operational kitchen]",Entire rental unit,No mencionado,2,"[Clean sheets, Extra towels, Soft pillows]","[Maxi grocery store, Convenience store, McDonald's restaurant, Promenade Masson, Parks, Community pool, Bus station, Biodome, Olympic Stadium]"


In [65]:
# ## 10. Ejercicio 1 - Guardado del resultado estructurado
#
# Guardamos el resultado de la extracción de entidades en un archivo CSV.
# Este archivo contiene el JSON estructurado convertido a tabla.
#
# Se guarda en la carpeta results para separar los resultados finales de los datos originales.

airbnb_entities_path = RESULTS_DIR / "ejercicio1_airbnb_entidades_descripciones.csv"

df_airbnb_entities.to_csv(airbnb_entities_path, index=False, encoding="utf-8-sig")

print("Resultado guardado en:", airbnb_entities_path)

Resultado guardado en: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\results\ejercicio1_airbnb_entidades_descripciones.csv


In [66]:
# ## 11. Ejercicio 1 - Selección de comentarios largos
#
# En esta parte se trabaja con el dataset Airbnb_reviews_5000.csv.
# El objetivo es analizar comentarios de usuarios sobre alojamientos.
#
# Para enviar al LLM comentarios con suficiente contenido, se seleccionan los 20 comentarios
# con mayor longitud en la columna "comments".
#
# También se eliminan comentarios vacíos para evitar llamadas al modelo con texto sin información.

df_reviews_analysis = df_reviews.copy()

df_reviews_analysis["comments"] = df_reviews_analysis["comments"].fillna("")
df_reviews_analysis["comment_length"] = df_reviews_analysis["comments"].str.len()

df_top_comments = (
    df_reviews_analysis
    .sort_values("comment_length", ascending=False)
    .head(20)
    .copy()
)

print("Número de comentarios seleccionados:", len(df_top_comments))
print("Longitud mínima seleccionada:", df_top_comments["comment_length"].min())
print("Longitud máxima seleccionada:", df_top_comments["comment_length"].max())

df_top_comments[
    [
        "name",
        "host_name",
        "date",
        "language",
        "comment_length",
        "comments"
    ]
].head()

Número de comentarios seleccionados: 20
Longitud mínima seleccionada: 1423
Longitud máxima seleccionada: 2976


,name,host_name,date,language,comment_length,comments
2000,FLH Porto Homey Flat with Balcony,Feels Like Home,2022-10-27,en,2976,"This WOULD have been a 5 star review BUT:<br/><br/>I don't typically leave negative or critical reviews but given the potential for this space to be incredible, I wanted to provide some feedback... the space is actually beautiful and well-equipped (with a working washer/dryer). It would be very..."
3876,Oporto FR Cativo Flat,Fábio,2015-05-18,en,2392,Two of us stayed in the apartment for a week in mid May 2015.\n<br/>\n<br/>Fábio greeted us at the door to the apartment building. There is a lift in the building which helped us to move our luggage up to the 2nd floor. Although it was only a short climb up two flights of stairs to the apartme...
2456,Terrace Duplex in Porto,João,2017-10-12,en,2226,"The location of the flat is great, few minutes walk to the metro station that takes you everywhere in Porto. Pedro kindly offered us an earlier check in since our flight arrived in the morning. He showed us how to get around and places to go and we've been to all of them! Thank you Pedro! <br/>T..."
1908,Belomonte River View Apartments 1,Belomonte River View Apartments,2022-06-29,en,2011,"Amazing views of the Douro River with Porto on the near side and Gaia on the far side. Even though we didn't spend all that much time at the apartment, whenever we were there we were usually checking out the views, the pigeons and sea gulls, or the group of cats on the rooftops and patios below...."
1636,Pinheiro Cardoso House - Room 4,Luís,2022-10-09,pt,1928,"Não recomendo! De todo! <br/>Este Airbnb é no “”“centro””” do Porto. Encontra-de de facto a 10 minutos da estação Campo 24 de Agosto mas acaba por ser num sítio que inspira pouca confiança. <br/>Ao chegarmos ao Airbnb, a chave do nosso quarto não se encontrava na caixa de segurança, apenas a de ..."


In [67]:
# ## 12. Ejercicio 1 - Prompt para análisis de comentarios
#
# En esta celda se define un prompt para analizar comentarios de usuarios sobre alojamientos.
# El objetivo es extraer información útil sobre limpieza, ubicación, calidad-precio,
# sentimiento general y otros aspectos relevantes.
#
# Se pide una salida en JSON para poder convertir fácilmente los resultados en un DataFrame.
# Si algún aspecto no aparece en el comentario, se debe indicar "No mencionado".

def build_airbnb_review_analysis_prompt(row):
    prompt = f"""
Eres un sistema experto en análisis de comentarios de usuarios sobre alojamientos turísticos.

Tu tarea es analizar el comentario de un huésped y devolver únicamente un objeto JSON válido.

Extrae los siguientes campos:

- accommodation_name: nombre del alojamiento
- review_sentiment: Positivo, Negativo, Neutral o Mixto
- cleanliness: opinión sobre la limpieza
- location: opinión sobre la ubicación
- value_for_money: opinión sobre la relación calidad-precio
- positive_aspects: lista de aspectos positivos mencionados
- negative_aspects: lista de aspectos negativos mencionados
- relevant_insights: lista de insights útiles para el anfitrión o para futuros huéspedes
- short_summary: resumen breve del comentario

Reglas:
1. Devuelve solo JSON válido, sin texto antes ni después.
2. No inventes información.
3. Si un campo no aparece en el comentario, usa "No mencionado".
4. Las listas deben ser listas JSON.
5. Mantén las respuestas breves y claras.

Información disponible:
- accommodation_name: {row.get("name", "No mencionado")}
- host_name: {row.get("host_name", "No mencionado")}
- date: {row.get("date", "No mencionado")}
- language: {row.get("language", "No mencionado")}

Comentario:
\"\"\"{row.get("comments", "")}\"\"\"

Formato exacto de salida:
{{
  "accommodation_name": "...",
  "review_sentiment": "...",
  "cleanliness": "...",
  "location": "...",
  "value_for_money": "...",
  "positive_aspects": ["...", "..."],
  "negative_aspects": ["...", "..."],
  "relevant_insights": ["...", "..."],
  "short_summary": "..."
}}
"""
    return prompt

In [68]:
# ## 13. Ejercicio 1 - Prueba del análisis de un comentario
#
# Antes de analizar todos los comentarios seleccionados, probamos el prompt con una sola fila.
# Esto permite comprobar que el modelo devuelve un JSON válido con los campos esperados.
#
# Si la respuesta es correcta, después aplicaremos el mismo proceso a los 20 comentarios.

test_comment_row = df_top_comments.iloc[0]

test_comment_prompt = build_airbnb_review_analysis_prompt(test_comment_row)

test_comment_response = get_completion(
    prompt=test_comment_prompt,
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=900
)

print(test_comment_response)

{
  "accommodation_name": "FLH Porto Homey Flat with Balcony",
  "review_sentiment": "Mixto",
  "cleanliness": "No mencionado",
  "location": "No mencionado",
  "value_for_money": "No mencionado",
  "positive_aspects": ["Beautiful and well-equipped space", "Working washer/dryer"],
  "negative_aspects": ["Poor management and maintenance", "Broken items in the apartment", "No shower soap or shampoo/conditioner provided", "Inconvenient key pickup process"],
  "relevant_insights": ["Management should improve maintenance and provide basic toiletries", "Consider changing management company for better guest experience"],
  "short_summary": "The space is beautiful but suffers from poor management and maintenance issues that detract from the overall experience."
}


In [69]:
# ## 14. Ejercicio 1 - Análisis de los 20 comentarios seleccionados
#
# En esta celda se aplica el prompt de análisis a los 20 comentarios más largos.
# El resultado se guarda como una lista de diccionarios JSON.
#
# Se añade time.sleep() entre llamadas para reducir el riesgo de problemas por límites de uso.
# También se conserva información original como el nombre del alojamiento, la fecha y el idioma.

airbnb_reviews_results = []

for index, row in df_top_comments.iterrows():
    print(f"Procesando comentario {len(airbnb_reviews_results) + 1} de {len(df_top_comments)}...")
    
    prompt = build_airbnb_review_analysis_prompt(row)
    
    response = get_completion(
        prompt=prompt,
        model="gpt-4o-mini",
        temperature=0.2,
        max_tokens=900
    )
    
    try:
        parsed_response = json.loads(response)
        parsed_response["original_name"] = row["name"]
        parsed_response["host_name"] = row["host_name"]
        parsed_response["date"] = row["date"]
        parsed_response["language"] = row["language"]
        airbnb_reviews_results.append(parsed_response)
    except json.JSONDecodeError:
        print("Error al convertir la respuesta a JSON en comentario index:", index)
        print(response)
    
    time.sleep(1)

print("Análisis completados:", len(airbnb_reviews_results))

Procesando comentario 1 de 20...
Procesando comentario 2 de 20...
Procesando comentario 3 de 20...
Procesando comentario 4 de 20...
Procesando comentario 5 de 20...
Procesando comentario 6 de 20...
Procesando comentario 7 de 20...
Procesando comentario 8 de 20...
Procesando comentario 9 de 20...
Procesando comentario 10 de 20...
Procesando comentario 11 de 20...
Procesando comentario 12 de 20...
Procesando comentario 13 de 20...
Procesando comentario 14 de 20...
Procesando comentario 15 de 20...
Procesando comentario 16 de 20...
Procesando comentario 17 de 20...
Procesando comentario 18 de 20...
Procesando comentario 19 de 20...
Procesando comentario 20 de 20...
Análisis completados: 20


In [70]:
# ## 15. Ejercicio 1 - Conversión del análisis de comentarios a DataFrame
#
# Las respuestas del LLM se convierten en un DataFrame.
# Esto permite revisar de forma tabular el sentimiento, limpieza, ubicación,
# relación calidad-precio, aspectos positivos, aspectos negativos e insights.
#
# Este DataFrame representa la salida estructurada de la segunda parte del Ejercicio 1.

df_airbnb_reviews_analysis = pd.DataFrame(airbnb_reviews_results)

columns_order_reviews = [
    "original_name",
    "accommodation_name",
    "host_name",
    "date",
    "language",
    "review_sentiment",
    "cleanliness",
    "location",
    "value_for_money",
    "positive_aspects",
    "negative_aspects",
    "relevant_insights",
    "short_summary"
]

df_airbnb_reviews_analysis = df_airbnb_reviews_analysis[columns_order_reviews]

df_airbnb_reviews_analysis.head()

,original_name,accommodation_name,host_name,date,language,review_sentiment,cleanliness,location,value_for_money,positive_aspects,negative_aspects,relevant_insights,short_summary
0,FLH Porto Homey Flat with Balcony,FLH Porto Homey Flat with Balcony,Feels Like Home,2022-10-27,en,Mixto,No mencionado,No mencionado,No mencionado,"[Beautiful and well-equipped space, Working washer/dryer]","[Poor management and maintenance, Broken items in the apartment, No shower soap or shampoo/conditioner provided, Inconvenient check-in process]","[Management should improve maintenance and provide basic toiletries, Consider a more convenient check-in process for guests]",The space is beautiful but suffers from poor management and maintenance issues that detract from the overall experience.
1,Oporto FR Cativo Flat,Oporto FR Cativo Flat,Fábio,2015-05-18,en,Positivo,No mencionado,"Bien ubicado, a 5 minutos a pie de la estación de metro y cerca del casco antiguo.",No mencionado,"[Fábio es un buen anfitrión y superó las expectativas., El apartamento ha sido recientemente renovado a un alto estándar., Bien equipado con utensilios de cocina., Wi-Fi y aire acondicionado funcionaron bien.]","[El baño es pequeño., La presión del agua no es equivalente a una ducha de potencia., El colchón es duro y puede requerir adaptación.]","[El apartamento es adecuado para una persona en el baño a la vez., El apartamento no se calienta demasiado durante el día., Hay buenas opciones de restaurantes y supermercados en la zona.]","El apartamento es cómodo, bien ubicado y el anfitrión es muy atento, aunque el baño es pequeño y el colchón puede ser duro."
2,Terrace Duplex in Porto,Terrace Duplex in Porto,João,2017-10-12,en,Mixto,No mencionado,"Great location, close to metro station",No mencionado,"[Great location, Helpful host, Nice cafe below, Spacious flat, Rooftop area]","[Shower curtain issue, Lack of basic cooking supplies, Uncomfortable sofa bed mattress, Need for maintenance]","[Consider providing basic cooking supplies, Replace the sofa bed mattress for better comfort, Address minor maintenance issues]","The flat has a great location and helpful host, but there are issues with the shower curtain, lack of cooking supplies, and an uncomfortable sofa bed."
3,Belomonte River View Apartments 1,Belomonte River View Apartments 1,Belomonte River View Apartments,2022-06-29,en,Mixto,No mencionado,"Bien ubicada, fácil acceso a los lugares de interés",No mencionado,"[Vistas increíbles del río Douro, Gran anfitrión, Buena ubicación para restaurantes, No necesitó transporte público]","[Ruido del tráfico en las habitaciones, Acceso al medio baño incómodo]","[Considerar insonorizar las habitaciones, Mejorar el acceso al medio baño]","Disfrutamos de las vistas y la ubicación, pero hubo ruido del tráfico y el acceso al medio baño fue incómodo."
4,Pinheiro Cardoso House - Room 4,Pinheiro Cardoso House - Room 4,Luís,2022-10-09,pt,Negativo,No mencionado,Inspira pouca confiança,Há muitos sítios melhores e mais em conta pela qualidade,[],"[Chave do quarto não estava na caixa de segurança, Recepção rude e desinformada, Quarto não corresponde às fotografias, Poliban mal vedado causando alagamento, Sem água quente, Falta de resposta do anfitrião]","[Melhorar a comunicação e atendimento na recepção, Verificar a limpeza e manutenção dos quartos antes do check-in, Garantir que as informações sobre a localização sejam claras e precisas]","Experiência negativa devido à falta de confiança na localização, problemas com a chave, atendimento rude e falta de água quente."


In [71]:
# ## 16. Ejercicio 1 - Guardado del análisis de comentarios
#
# Guardamos el resultado del análisis de comentarios en un archivo CSV.
# Este archivo es el resultado final de la segunda parte del Ejercicio 1.
#
# Se guarda en la carpeta results para mantener separados los resultados finales.

airbnb_reviews_analysis_path = RESULTS_DIR / "ejercicio1_airbnb_analisis_comentarios.csv"

df_airbnb_reviews_analysis.to_csv(
    airbnb_reviews_analysis_path,
    index=False,
    encoding="utf-8-sig"
)

print("Resultado guardado en:", airbnb_reviews_analysis_path)

Resultado guardado en: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\results\ejercicio1_airbnb_analisis_comentarios.csv


In [72]:
# ------------------------------- Ejercicio 2 -----------------------------------

# ## 17. Ejercicio 2 - Carga del dataset de reseñas de videojuegos
#
# En esta sección se carga el dataset principal del entregable final.
# Cada fila representa una reseña de un videojuego.
#
# Según el enunciado, el archivo contiene:
# - Contenido: texto libre de la reseña.
# - Valoración: Recomendado o No recomendado.
# - Recomendado binario: 1 si es recomendado, 0 si no es recomendado.
#
# Antes de aplicar prompts, revisamos la estructura real del CSV.

videogames_path = DATA_RAW / "videogames_reviews.csv"

df_videogames = pd.read_csv(videogames_path)

print("Dataset videogames_reviews.csv")
print("Filas:", df_videogames.shape[0])
print("Columnas:", df_videogames.shape[1])
print()

print("Columnas:")
print(df_videogames.columns.tolist())

df_videogames.head()

Dataset videogames_reviews.csv
Filas: 20000
Columnas: 4

Columnas:
['Unnamed: 0', 'Contenido', 'Valoración', 'Recomendado_binario']


,Unnamed: 0,Contenido,Valoración,Recomendado_binario
0,0,2 marzo so bad,No recomendado,0
1,1,10 febrero actualmente recomiendo juego contaba requisitos necesarios poder jugarlo llevado disgusto grande tenia muchas ganas jugarlo dececpionado primera hora juego medianamente jugable llegas hogwarts practicamente injugable configures veas videos tutoriales fps van bajar va ser imposible jug...,No recomendado,0
2,2,9 febrero increíblemente gracioso ver cómo cdpr después catástrofe cyberpunk 2077 esfuerza cada vez tomar decisiones estúpidas vez tocado juego reconocimiento dio día espero ganas caída empresa diabloah agregar launcher abrir juego quizás gracia hace menos obligan instalarlo abrir juego hace ubi...,No recomendado,0
3,3,the world in this game is extremely static there is almost interaction you can do the npc ai is sometimes worse than in morrowind speaking of elder scrolls that series learned long time ago its bad idea to have skills level with jumping or running but not this cp for example to level athletics y...,No recomendado,0
4,4,zero replayability i finished this game in about 100 hours and pretty much 100ed the game after that there is literally nothing to do except just drive around and shoot people tried to play through second time month ago and its rough knowing that your choices in the game dont mean jack lot of pe...,No recomendado,0


In [73]:
# ## 18. Ejercicio 2 - Selección de las 100 reseñas más largas
#
# El enunciado indica que se parte de un DataFrame filtrado previamente con las 100 reseñas
# con mayor longitud de contenido.
#
# Como estamos construyendo el notebook completo, reproducimos ese paso aquí.
# Para ello calculamos la longitud del texto de la columna "Contenido" y seleccionamos
# las 100 reseñas más largas.
#
# También renombramos la columna "Unnamed: 0" como "Id" para tener un identificador claro
# de cada reseña.

df_videogames_work = df_videogames.copy()

df_videogames_work = df_videogames_work.rename(columns={"Unnamed: 0": "Id"})

df_videogames_work["Contenido"] = df_videogames_work["Contenido"].fillna("")
df_videogames_work["content_length"] = df_videogames_work["Contenido"].str.len()

df_top_100_reviews = (
    df_videogames_work
    .sort_values("content_length", ascending=False)
    .head(100)
    .copy()
)

print("Número de reseñas seleccionadas:", len(df_top_100_reviews))
print("Longitud mínima seleccionada:", df_top_100_reviews["content_length"].min())
print("Longitud máxima seleccionada:", df_top_100_reviews["content_length"].max())

df_top_100_reviews[
    [
        "Id",
        "Contenido",
        "Valoración",
        "Recomendado_binario",
        "content_length"
    ]
].head()

Número de reseñas seleccionadas: 100
Longitud mínima seleccionada: 5421
Longitud máxima seleccionada: 7972


,Id,Contenido,Valoración,Recomendado_binario,content_length
19502,19502,suiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiisuiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiiii...,Recomendado,1,7972
17660,17660,this was probably my first preorder i felt that if the game turned out to be the worst game i played or didnt even run 10 dollars i paid for the witcher 2 and the witcher 3 felt so much like stealing that the extra 60 would still not cover the amount i would pay for the enjoyment i got from them...,Recomendado,1,7662
7187,7187,oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty biased regarding cdpr and their games and thats why its painful to admit and accept the fact that in its current state cp 2077 is very dissapointing by cdpr standards and in gener...,No recomendado,0,7638
4187,4187,oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty biased regarding cdpr and their games and thats why its painful to admit and accept the fact that in its current state cp 2077 is very dissapointing by cdpr standards and in gener...,No recomendado,0,7638
5415,5415,i know many will handwave away any criticisms of this game as salt or reply with git gud but as fan of fromsoft games since demons souls this game felt like the most punitive and least satisfying of the games while the setting is undoubtedly beautiful and there is fascinating story and interesti...,No recomendado,0,7609


In [74]:
# ## 19. Ejercicio 2 - Prompt para filtrar reseñas relevantes
#
# El primer paso del pipeline con LLM consiste en decidir si una reseña es relevante o no.
#
# Una reseña se considera relevante si contiene opiniones, argumentos, aspectos positivos,
# aspectos negativos o información útil sobre la experiencia del jugador.
#
# Una reseña se considera no relevante si es spam, texto repetitivo, contenido vacío,
# ruido, bromas sin información útil o texto que no permite extraer insights.
#
# El modelo debe devolver un JSON para cada reseña, indicando si es relevante,
# el motivo de la decisión y una breve justificación.

def build_videogame_relevance_prompt(row):
    prompt = f"""
Eres un sistema experto en análisis de reseñas de videojuegos.

Tu tarea es decidir si la siguiente reseña es relevante para un análisis posterior con LLM.

Una reseña relevante debe cumplir al menos una de estas condiciones:
- Contiene una opinión clara sobre el videojuego.
- Menciona aspectos positivos o negativos.
- Habla de jugabilidad, historia, gráficos, rendimiento, dificultad, bugs, contenido, precio o experiencia del jugador.
- Permite extraer insights útiles.

Una reseña no relevante incluye casos como:
- Texto repetitivo sin significado.
- Spam.
- Comentarios vacíos o casi vacíos.
- Bromas sin información útil.
- Texto que no permite extraer ninguna opinión clara.

Devuelve únicamente un objeto JSON válido, sin explicaciones antes ni después.

Datos de la reseña:
- Id: {row.get("Id", "No mencionado")}
- Valoración original: {row.get("Valoración", "No mencionado")}
- Recomendado_binario: {row.get("Recomendado_binario", "No mencionado")}

Reseña:
\"\"\"{row.get("Contenido", "")}\"\"\"

Formato exacto de salida:
{{
  "Id": "{row.get("Id", "No mencionado")}",
  "is_relevant": true,
  "relevance_reason": "explicación breve",
  "review_type": "Informativa | Parcialmente informativa | No informativa"
}}
"""
    return prompt

In [75]:
# ## 20. Ejercicio 2 - Prueba del prompt de filtrado
#
# Antes de procesar las 100 reseñas, probamos el prompt con una sola reseña.
# Esto permite comprobar si el LLM clasifica correctamente una reseña larga pero poco informativa.
#
# La primera reseña seleccionada contiene mucho texto repetitivo, por lo que debería marcarse
# como no relevante o parcialmente informativa.

test_review_row = df_top_100_reviews.iloc[0]

test_relevance_prompt = build_videogame_relevance_prompt(test_review_row)

test_relevance_response = get_completion(
    prompt=test_relevance_prompt,
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=400
)

print(test_relevance_response)

{
  "Id": "19502",
  "is_relevant": false,
  "relevance_reason": "Texto repetitivo sin significado y sin opiniones claras.",
  "review_type": "No informativa"
}


In [76]:
# ## 21. Ejercicio 2 - Filtrado de relevancia para las 100 reseñas
#
# En esta celda se aplica el primer LLM a las 100 reseñas más largas.
# El objetivo es decidir cuáles contienen información útil para el análisis posterior.
#
# Este paso corresponde al filtrado de contenido relevante mediante prompting.
# Se usa time.sleep() para reducir el riesgo de alcanzar límites de uso de la API.
#
# El resultado será una lista de JSON con:
# - Id
# - is_relevant
# - relevance_reason
# - review_type

videogame_relevance_results = []

for index, row in df_top_100_reviews.iterrows():
    print(f"Procesando reseña {len(videogame_relevance_results) + 1} de {len(df_top_100_reviews)}...")
    
    prompt = build_videogame_relevance_prompt(row)
    
    response = get_completion(
        prompt=prompt,
        model="gpt-4o-mini",
        temperature=0.2,
        max_tokens=400
    )
    
    try:
        parsed_response = json.loads(response)
        parsed_response["Id"] = int(parsed_response["Id"])
        videogame_relevance_results.append(parsed_response)
    except json.JSONDecodeError:
        print("Error al convertir la respuesta a JSON en Id:", row["Id"])
        print(response)
    
    time.sleep(1)

print("Filtrados completados:", len(videogame_relevance_results))

Procesando reseña 1 de 100...
Procesando reseña 2 de 100...
Procesando reseña 3 de 100...
Procesando reseña 4 de 100...
Procesando reseña 5 de 100...
Procesando reseña 6 de 100...
Procesando reseña 7 de 100...
Procesando reseña 8 de 100...
Procesando reseña 9 de 100...
Procesando reseña 10 de 100...
Procesando reseña 11 de 100...
Procesando reseña 12 de 100...
Procesando reseña 13 de 100...
Procesando reseña 14 de 100...
Procesando reseña 15 de 100...
Procesando reseña 16 de 100...
Procesando reseña 17 de 100...
Procesando reseña 18 de 100...
Procesando reseña 19 de 100...
Procesando reseña 20 de 100...
Procesando reseña 21 de 100...
Procesando reseña 22 de 100...
Procesando reseña 23 de 100...
Procesando reseña 24 de 100...
Procesando reseña 25 de 100...
Procesando reseña 26 de 100...
Procesando reseña 27 de 100...
Procesando reseña 28 de 100...
Procesando reseña 29 de 100...
Procesando reseña 30 de 100...
Procesando reseña 31 de 100...
Procesando reseña 32 de 100...
Procesando reseña

In [77]:
# ## 22. Ejercicio 2 - Creación del DataFrame de reseñas relevantes
#
# Convertimos los resultados del primer LLM en un DataFrame.
# Después los unimos con las 100 reseñas originales usando la columna Id.
#
# El objetivo es crear un nuevo DataFrame reducido que contenga solo las reseñas
# que el modelo ha considerado relevantes para el análisis posterior.

df_relevance = pd.DataFrame(videogame_relevance_results)

df_relevance.head()

,Id,is_relevant,relevance_reason,review_type
0,19502,False,La reseña consiste en texto repetitivo sin significado y no contiene una opinión clara sobre el videojuego.,No informativa
1,17660,True,"La reseña contiene opiniones claras sobre la jugabilidad, la historia, los gráficos y la experiencia del jugador en Cyberpunk 2077, así como aspectos positivos y negativos del juego.",Informativa
2,7187,True,"La reseña contiene una opinión clara sobre el videojuego, menciona aspectos negativos como bugs, falta de contenido y problemas de jugabilidad, así como aspectos positivos como los gráficos y la atmósfera.",Informativa
3,4187,True,"La reseña contiene una opinión clara sobre el juego, menciona aspectos negativos como bugs, falta de contenido y una historia decepcionante, así como detalles sobre la jugabilidad y gráficos.",Informativa
4,5415,True,"La reseña contiene opiniones claras sobre la jugabilidad, la dificultad, el diseño de los enemigos y la falta de recompensa, lo que permite extraer insights útiles sobre la experiencia del jugador.",Informativa


In [78]:
# Unimos la decisión de relevancia con el DataFrame original de las 100 reseñas.
# Así conservamos el contenido completo de cada reseña junto con la clasificación del LLM.

df_top_100_with_relevance = df_top_100_reviews.merge(
    df_relevance,
    on="Id",
    how="left"
)

df_relevant_reviews = df_top_100_with_relevance[
    df_top_100_with_relevance["is_relevant"] == True
].copy()

print("Reseñas iniciales:", len(df_top_100_reviews))
print("Reseñas relevantes:", len(df_relevant_reviews))
print("Reseñas eliminadas:", len(df_top_100_reviews) - len(df_relevant_reviews))

df_relevant_reviews[
    [
        "Id",
        "Valoración",
        "Recomendado_binario",
        "content_length",
        "is_relevant",
        "review_type",
        "relevance_reason",
        "Contenido"
    ]
].head()

Reseñas iniciales: 100
Reseñas relevantes: 99
Reseñas eliminadas: 1


,Id,Valoración,Recomendado_binario,content_length,is_relevant,review_type,relevance_reason,Contenido
1,17660,Recomendado,1,7662,True,Informativa,"La reseña contiene opiniones claras sobre la jugabilidad, la historia, los gráficos y la experiencia del jugador en Cyberpunk 2077, así como aspectos positivos y negativos del juego.",this was probably my first preorder i felt that if the game turned out to be the worst game i played or didnt even run 10 dollars i paid for the witcher 2 and the witcher 3 felt so much like stealing that the extra 60 would still not cover the amount i would pay for the enjoyment i got from them...
2,7187,No recomendado,0,7638,True,Informativa,"La reseña contiene una opinión clara sobre el videojuego, menciona aspectos negativos como bugs, falta de contenido y problemas de jugabilidad, así como aspectos positivos como los gráficos y la atmósfera.",oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty biased regarding cdpr and their games and thats why its painful to admit and accept the fact that in its current state cp 2077 is very dissapointing by cdpr standards and in gener...
3,4187,No recomendado,0,7638,True,Informativa,"La reseña contiene una opinión clara sobre el juego, menciona aspectos negativos como bugs, falta de contenido y una historia decepcionante, así como detalles sobre la jugabilidad y gráficos.",oh well admittedly its difficult for to write this review let start by saying that i was and kind of still am pretty biased regarding cdpr and their games and thats why its painful to admit and accept the fact that in its current state cp 2077 is very dissapointing by cdpr standards and in gener...
4,5415,No recomendado,0,7609,True,Informativa,"La reseña contiene opiniones claras sobre la jugabilidad, la dificultad, el diseño de los enemigos y la falta de recompensa, lo que permite extraer insights útiles sobre la experiencia del jugador.",i know many will handwave away any criticisms of this game as salt or reply with git gud but as fan of fromsoft games since demons souls this game felt like the most punitive and least satisfying of the games while the setting is undoubtedly beautiful and there is fascinating story and interesti...
5,7751,No recomendado,0,7609,True,Informativa,"La reseña contiene una opinión clara y detallada sobre el videojuego, mencionando aspectos negativos como el diseño, la historia, la jugabilidad y los problemas técnicos.",i had enormous expectations having put thousand hours into dark souls 3 alone sadly er is tainted by design decisions not fully thought through and refusal to question the ideals of framework first established with demons souls in 2009 and i ended it wishing i hadnt played it at all more on this...


In [79]:
# ## 23. Ejercicio 2 - Guardado de reseñas relevantes
#
# Guardamos el DataFrame reducido después del filtrado de relevancia.
# Este archivo representa la salida del primer paso del pipeline con LLM.
#
# A partir de este DataFrame se realizará la segunda fase:
# extracción estructurada de información con otro prompt/modelo.

relevant_reviews_path = DATA_PROCESSED / "ejercicio2_resenas_videojuegos_relevantes.csv"

df_relevant_reviews.to_csv(
    relevant_reviews_path,
    index=False,
    encoding="utf-8-sig"
)

print("Reseñas relevantes guardadas en:", relevant_reviews_path)

Reseñas relevantes guardadas en: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\data\processed\ejercicio2_resenas_videojuegos_relevantes.csv


In [80]:
# ## 24. Ejercicio 2 - Prompt para extracción estructurada de reseñas de videojuegos
#
# Esta es la segunda fase del pipeline con LLMs.
# A partir de las reseñas ya filtradas como relevantes, se extrae información estructurada.
#
# El enunciado pide extraer al menos 3 entidades o atributos relevantes.
# En este trabajo se extraen varios atributos: sentimiento general, aspectos positivos,
# aspectos negativos, dificultad, rendimiento técnico, bugs, jugabilidad e insight principal.
#
# Se pide una salida JSON para poder convertir los resultados fácilmente en un DataFrame.

def build_videogame_extraction_prompt(row):
    prompt = f"""
Eres un sistema experto en análisis de reseñas de videojuegos.

Tu tarea es analizar una reseña relevante y extraer información estructurada.

Devuelve únicamente un objeto JSON válido, sin explicaciones antes ni después.

Campos a extraer:

- Id: identificador de la reseña.
- SentimientoGeneral: Positivo, Negativo, Neutral o Mixto.
- AspectosPositivos: lista de aspectos positivos mencionados.
- AspectosNegativos: lista de aspectos negativos mencionados.
- Dificultad: Demasiado Fácil, Fácil, Equilibrado, Difícil, Demasiado Difícil o No Mencionado.
- RendimientoTecnico: opinión sobre rendimiento, FPS, optimización, bugs técnicos o estabilidad.
- BugsProblemas: lista de bugs o problemas mencionados.
- Jugabilidad: opinión sobre mecánicas, combate, controles, exploración o experiencia de juego.
- HistoriaContenido: opinión sobre historia, misiones, personajes, contenido o mundo del juego.
- PrecioValor: opinión sobre precio, valor, cantidad de contenido o relación calidad-precio.
- InsightPrincipal: conclusión breve y útil de la reseña.
- RecomendadoOriginal: valoración original del dataset.
- RecomendadoBinario: valor binario original del dataset.

Reglas:
1. No inventes información.
2. Si un aspecto no aparece, usa "No mencionado".
3. Las listas deben ser listas JSON.
4. Mantén las respuestas breves y claras.
5. La salida debe ser JSON válido.

Datos de la reseña:
- Id: {row.get("Id", "No mencionado")}
- Valoración original: {row.get("Valoración", "No mencionado")}
- Recomendado_binario: {row.get("Recomendado_binario", "No mencionado")}

Reseña:
\"\"\"{row.get("Contenido", "")}\"\"\"

Formato exacto de salida:
{{
  "Id": "{row.get("Id", "No mencionado")}",
  "SentimientoGeneral": "...",
  "AspectosPositivos": ["...", "..."],
  "AspectosNegativos": ["...", "..."],
  "Dificultad": "...",
  "RendimientoTecnico": "...",
  "BugsProblemas": ["...", "..."],
  "Jugabilidad": "...",
  "HistoriaContenido": "...",
  "PrecioValor": "...",
  "InsightPrincipal": "...",
  "RecomendadoOriginal": "{row.get("Valoración", "No mencionado")}",
  "RecomendadoBinario": "{row.get("Recomendado_binario", "No mencionado")}"
}}
"""
    return prompt

In [81]:
# ## 25. Ejercicio 2 - Prueba de extracción estructurada
#
# Antes de procesar todas las reseñas relevantes, probamos el segundo prompt con una sola reseña.
# Esto permite comprobar que el modelo devuelve un JSON válido con los campos esperados.
#
# Si el resultado es correcto, después aplicaremos este mismo prompt al DataFrame filtrado.

test_extraction_row = df_relevant_reviews.iloc[0]

test_extraction_prompt = build_videogame_extraction_prompt(test_extraction_row)

test_extraction_response = get_completion(
    prompt=test_extraction_prompt,
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=1200
)

print(test_extraction_response)

{
  "Id": "17660",
  "SentimientoGeneral": "Positivo",
  "AspectosPositivos": [
    "Detalles y vida en la ciudad",
    "Mecánicas de juego variadas",
    "Historia y personajes bien desarrollados",
    "Libertad en la jugabilidad",
    "Innovaciones en la narrativa"
  ],
  "AspectosNegativos": [
    "Dificultad a veces desequilibrada",
    "Falta de cierre en algunas misiones"
  ],
  "Dificultad": "Equilibrado",
  "RendimientoTecnico": "Estable FPS en GTX 1070",
  "BugsProblemas": ["No mencionado"],
  "Jugabilidad": "Mecánicas de juego variadas y exploración divertida",
  "HistoriaContenido": "Historia y temas bien ejecutados, aunque con falta de cierre",
  "PrecioValor": "Vale el precio completo por la experiencia",
  "InsightPrincipal": "Cyberpunk 2077 ofrece una experiencia única y rica en detalles, a pesar de algunos problemas menores.",
  "RecomendadoOriginal": "Recomendado",
  "RecomendadoBinario": "1"
}


In [82]:
# ## 26. Ejercicio 2 - Extracción estructurada de todas las reseñas relevantes
#
# En esta celda se aplica el segundo prompt a todas las reseñas filtradas como relevantes.
# Esta es la fase principal de extracción de información estructurada.
#
# Se usa time.sleep() entre llamadas para gestionar posibles límites de uso de la API.
# El resultado será una lista de objetos JSON, uno por cada reseña relevante.

videogame_extraction_results = []

for index, row in df_relevant_reviews.iterrows():
    print(f"Procesando reseña relevante {len(videogame_extraction_results) + 1} de {len(df_relevant_reviews)}...")
    
    prompt = build_videogame_extraction_prompt(row)
    
    response = get_completion(
        prompt=prompt,
        model="gpt-4o-mini",
        temperature=0.2,
        max_tokens=1200
    )
    
    try:
        parsed_response = json.loads(response)
        parsed_response["Id"] = int(parsed_response["Id"])
        videogame_extraction_results.append(parsed_response)
    except json.JSONDecodeError:
        print("Error al convertir la respuesta a JSON en Id:", row["Id"])
        print(response)
    
    time.sleep(1)

print("Extracciones completadas:", len(videogame_extraction_results))

Procesando reseña relevante 1 de 99...
Procesando reseña relevante 2 de 99...
Procesando reseña relevante 3 de 99...
Procesando reseña relevante 4 de 99...
Procesando reseña relevante 5 de 99...
Procesando reseña relevante 6 de 99...
Procesando reseña relevante 7 de 99...
Procesando reseña relevante 8 de 99...
Procesando reseña relevante 9 de 99...
Procesando reseña relevante 10 de 99...
Procesando reseña relevante 11 de 99...
Procesando reseña relevante 12 de 99...
Procesando reseña relevante 13 de 99...
Procesando reseña relevante 14 de 99...
Procesando reseña relevante 15 de 99...
Procesando reseña relevante 16 de 99...
Procesando reseña relevante 17 de 99...
Procesando reseña relevante 18 de 99...
Procesando reseña relevante 19 de 99...
Procesando reseña relevante 20 de 99...
Procesando reseña relevante 21 de 99...
Procesando reseña relevante 22 de 99...
Procesando reseña relevante 23 de 99...
Procesando reseña relevante 24 de 99...
Procesando reseña relevante 25 de 99...
Procesand

In [83]:
# ## 27. Ejercicio 2 - Conversión de resultados estructurados a DataFrame
#
# Las respuestas del segundo LLM se han guardado como una lista de diccionarios.
# En esta celda se convierten en un DataFrame final.
#
# Este DataFrame representa el resultado estructurado del análisis de reseñas de videojuegos.

df_videogame_structured = pd.DataFrame(videogame_extraction_results)

columns_order_videogames = [
    "Id",
    "SentimientoGeneral",
    "AspectosPositivos",
    "AspectosNegativos",
    "Dificultad",
    "RendimientoTecnico",
    "BugsProblemas",
    "Jugabilidad",
    "HistoriaContenido",
    "PrecioValor",
    "InsightPrincipal",
    "RecomendadoOriginal",
    "RecomendadoBinario"
]

df_videogame_structured = df_videogame_structured[columns_order_videogames]

print("Filas del resultado final:", len(df_videogame_structured))
df_videogame_structured.head()

Filas del resultado final: 99


,Id,SentimientoGeneral,AspectosPositivos,AspectosNegativos,Dificultad,RendimientoTecnico,BugsProblemas,Jugabilidad,HistoriaContenido,PrecioValor,InsightPrincipal,RecomendadoOriginal,RecomendadoBinario
0,17660,Positivo,"[Detalles y vida en la ciudad, Libertad en las misiones, Historia y personajes bien desarrollados, Innovaciones en la jugabilidad]","[Dificultad a veces desequilibrada, Falta de cierre en algunas misiones]",Equilibrado,Estable FPS con GTX 1070,[No mencionado],Mecánicas variadas y exploración divertida,Temas profundos y bien ejecutados,Vale el precio completo por la experiencia,"Cyberpunk 2077 ofrece una experiencia única y rica en detalles, a pesar de algunos problemas menores.",Recomendado,1
1,7187,Negativo,"[Gráficos sólidos, Buena atmósfera, Música y banda sonora]","[Estado del juego decepcionante, Falta de mecánicas, Historia poco interesante, Repetitividad en misiones, Limitadas opciones de personalización]",No Mencionado,"Rendimiento estable al principio, pero con bugs que se volvieron más frecuentes","[Glitches gráficos, Animaciones extrañas, Texturas faltantes, Coches desapareciendo, Misiones sin poder completar]","Mecánicas y controles aceptables, pero falta de contenido significativo","Historia decepcionante hasta ahora, con misiones principales poco interesantes","No recomendado, se sugiere esperar un año para mejoras",Cyberpunk 2077 se siente incompleto y por debajo de los estándares de CDPR.,No recomendado,0
2,4187,Negativo,"[Gráficos sólidos, Buena atmósfera, Música y banda sonora]","[Bugs frecuentes y severos, Historia poco interesante, Mecánicas faltantes, Contenido repetitivo]",No Mencionado,Rendimiento estable con algunos problemas de bugs,"[Glitches gráficos, Animaciones extrañas, Texturas faltantes, Coches desapareciendo, Quests no finalizables]","Mecánicas y controles aceptables, pero falta de opciones de personalización","Historia decepcionante y no interesante, con misiones principales arruinadas por trailers",No Mencionado,El juego se siente incompleto y por debajo de los estándares de CDPR.,No recomendado,0
3,5415,Negativo,"[Hermoso entorno, Gran personalización, Momentos de emoción]","[Diseño de jefes frustrante, Sensación de falta de recompensa, Dificultad desproporcionada, Reutilización de enemigos y jefes]",Difícil,Problemas de rendimiento en la versión de PC,"[Caballo muere sin razón, Barras de salud de enemigos se rellenan extrañamente]",Mecánicas frustrantes y poco satisfactorias,Historia interesante con personajes fascinantes,No mencionado,"El juego es frustrante y poco gratificante, a pesar de algunos buenos elementos.",No recomendado,0
4,7751,Negativo,"[Combat responsive, Diverse enemy weapon and ability variety, Geographically impressive world, Creative character design]","[Poor design decisions, Repetitive content, Unbalanced weapon classes, Inconsistent magic, Bad camera, Technical issues, Boring quests, Disappointing story]",Equilibrado,Graphically antiquated with shader compilation stutter and disappointing framerates,"[Crashing, Cheating, Long hit registration times, Disconnects]",Missed opportunity with unbalanced combat and repetitive gameplay,Shockingly conservative with recycled plot points and less emotional investment,No mencionado,Elden Ring is a missed opportunity that fails to innovate and deliver a compelling story.,No recomendado,0


In [84]:
# ## 28. Ejercicio 2 - Guardado del resultado final estructurado
#
# Guardamos el resultado final del Ejercicio 2 en formato CSV.
# Este archivo contiene la información estructurada extraída de las reseñas relevantes.
#
# También guardamos una copia en data/processed y otra en results para facilitar la entrega.

videogame_results_path = RESULTS_DIR / "ejercicio2_analisis_videojuegos_resultado_final.csv"
videogame_processed_path = DATA_PROCESSED / "ejercicio2_analisis_videojuegos_resultado_final.csv"

df_videogame_structured.to_csv(
    videogame_results_path,
    index=False,
    encoding="utf-8-sig"
)

df_videogame_structured.to_csv(
    videogame_processed_path,
    index=False,
    encoding="utf-8-sig"
)

print("Resultado final guardado en:", videogame_results_path)
print("Copia guardada en:", videogame_processed_path)

Resultado final guardado en: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\results\ejercicio2_analisis_videojuegos_resultado_final.csv
Copia guardada en: d:\2_UPSKILL\1_2_PONTIA\8_Prompt_Engineering\trabajo_final_prompt_engineering\data\processed\ejercicio2_analisis_videojuegos_resultado_final.csv


In [85]:
# ## 29. Justificación técnica del pipeline de Prompt Engineering con LLMs
#
# En este trabajo se ha utilizado un pipeline de Prompt Engineering con LLMs basado en dos prompts principales,
# tal como pide el enunciado.
#
# Fase 1: filtrado de reseñas relevantes
# --------------------------------------
# Primero se seleccionaron las 100 reseñas con mayor longitud de contenido.
# Esta decisión viene indicada en el enunciado y permite trabajar con textos que potencialmente
# contienen más información útil.
#
# Sin embargo, una reseña larga no siempre es relevante. Por ejemplo, puede contener texto repetido,
# spam o contenido sin significado. Por eso se diseñó un primer prompt cuyo objetivo era clasificar
# cada reseña como relevante o no relevante.
#
# El primer LLM devuelve un JSON con:
# - Id
# - is_relevant
# - relevance_reason
# - review_type
#
# Este paso reduce el ruido del dataset antes de aplicar una extracción más detallada.
#
# Fase 2: extracción estructurada
# -------------------------------
# A partir de las reseñas consideradas relevantes, se aplicó un segundo prompt para extraer
# información estructurada.
#
# El segundo LLM devuelve un JSON con atributos como:
# - SentimientoGeneral
# - AspectosPositivos
# - AspectosNegativos
# - Dificultad
# - RendimientoTecnico
# - BugsProblemas
# - Jugabilidad
# - HistoriaContenido
# - PrecioValor
# - InsightPrincipal
# - RecomendadoOriginal
# - RecomendadoBinario
#
# Esta separación en dos tareas permite que cada prompt tenga un objetivo claro:
# primero decidir si la reseña sirve para el análisis, y después extraer información útil.
#
# Modelo utilizado
# ----------------
# Se ha utilizado el modelo gpt-4o-mini mediante la API de OpenAI.
# La elección se justifica porque es un modelo rápido, económico y suficiente para tareas
# de clasificación, extracción de información y generación de JSON estructurado.
#
# Parámetros utilizados
# ---------------------
# Se ha usado temperature=0.2 porque el objetivo del trabajo no es generar texto creativo,
# sino obtener respuestas consistentes, controladas y fáciles de convertir a DataFrame.
#
# También se han limitado los max_tokens en cada llamada:
# - 400 tokens para el filtrado de relevancia.
# - 1200 tokens para la extracción estructurada.
#
# Esto ayuda a controlar la longitud de la respuesta y el coste de las llamadas.
#
# Gestión de llamadas y rate limits
# ---------------------------------
# Las llamadas al modelo se han realizado fila por fila.
# Aunque el enunciado menciona la posibilidad de batching, se ha elegido procesar una reseña
# por llamada porque cada reseña es larga y porque así se reduce el riesgo de superar límites
# de tokens por petición.
#
# Se ha usado time.sleep(1) entre llamadas para espaciar las peticiones a la API.
# Esta decisión ayuda a evitar errores por rate limits y hace que el proceso sea más estable.
#
# Resultado final
# ---------------
# El resultado final se ha guardado en:
# - results/ejercicio2_analisis_videojuegos_resultado_final.csv
# - data/processed/ejercicio2_analisis_videojuegos_resultado_final.csv
#
# El CSV final contiene una tabla estructurada con los insights extraídos de las reseñas relevantes.

In [86]:
# ## 30. Conclusión final
#
# En este trabajo se han aplicado técnicas de Prompt Engineering para transformar texto no estructurado
# en información organizada y útil.
#
# En el Ejercicio 1 se han utilizado prompts para extraer entidades desde descripciones de alojamientos
# y para analizar comentarios de usuarios. Los resultados se han estructurado en DataFrames y se han
# guardado como archivos CSV.
#
# En el Ejercicio 2 se ha diseñado un pipeline de Prompt Engineering con LLMs basado en dos prompts principales.
# El primer prompt filtra reseñas relevantes y elimina contenido poco informativo.
# El segundo prompt extrae información estructurada de las reseñas relevantes.
#
# Esta separación de tareas mejora el control del proceso, reduce el ruido y facilita la obtención
# de insights como sentimiento general, aspectos positivos, aspectos negativos, dificultad,
# rendimiento técnico, bugs, jugabilidad, historia, precio/valor y recomendación original.
#
# El resultado final queda disponible como DataFrame dentro del notebook y también como archivo CSV
# para su revisión o entrega.